# 残差网络

欢迎来到本周的第一个作业！你将使用残差网络 (ResNets) 构建一个非常深的卷积网络。理论上，非常深的网络可以表示非常复杂的函数；但在实践中，它们很难训练。 [He et al.](https://arxiv.org/pdf/1512.03385.pdf) 引入的残差网络允许你训练比以前可行的更深的网络。

**在本作业结束时，你将能够：**

- 使用 Keras 在深度神经网络中实现 ResNet 的基本构建块
- 将这些构建块放在一起来实现和训练最先进的图像分类神经网络
- 在你的网络中实现跳跃连接

对于此作业，你将使用 Keras。 

在讨论问题之前，请运行下面的单元格来加载所需的包。

## 关于提交至 AutoGrader 的重要说明

向 AutoGrader 提交作业前，请确认满足以下要求：

1. 未在作业中添加 _额外的_  `print` 语句。
2. 未在作业中添加 _额外的_ 代码单元格。
3. 未修改任何函数的参数。
4. 未在计分练习中使用全局变量。除非明确要求使用全局变量，否则应使用局部变量。
5. 未修改无需改动的作业代码，例如创建 _额外的_ 变量。

如果违反上述要求，提交作业时可能出现 `Grader Error: Grader feedback not found` 或其他意外错误。在求助或调试作业之前，请先检查这些情况。如果不记得自己做过哪些改动，可以按照 [说明](https://www.coursera.org/learn/convolutional-neural-networks/supplement/DS4yP/h-ow-to-refresh-your-workspace) 获取一份新的作业。

## 目录

- [1 - 依赖库](#1)
- [2 - 极深神经网络的问题](#2)
- [3 - 构建残差网络](#3)
    - [3.1 - 恒等块](#3-1)
        - [练习 1 - identity_block](#ex-1)
    - [3.2 - 卷积块](#3-2)
        - [练习 2 - convolutional_block](#ex-2)
- [4 - 构建你的第一个 ResNet 模型（50 层）](#4)
    - [练习 3 - ResNet50](#ex-3)
- [5 - 在你自己的图像 (Optional/Ungraded) 上进行测试](#5)
- [6 - 参考文献](#6)

<a name='1'></a>
## 1 - 依赖库

In [ ]:
### v3.0

In [ ]:
import tensorflow as tf
import numpy as np
import scipy.misc
from tensorflow.keras.applications.resnet_v2 import ResNet50V2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet_v2 import preprocess_input, decode_predictions
from tensorflow.keras import layers
from tensorflow.keras.layers import Input, Add, Dense, Activation, ZeroPadding2D, Flatten, Conv2D, AveragePooling2D, MaxPooling2D, GlobalMaxPooling2D
from tensorflow.keras.models import Model, load_model
from resnets_utils import *
from tensorflow.keras.initializers import random_uniform, glorot_uniform, constant, identity
from tensorflow.python.framework.ops import EagerTensor
from matplotlib.pyplot import imshow


from test_utils import summary, comparator
import public_tests

%matplotlib inline
np.random.seed(1)
tf.random.set_seed(2)

<a name='2'></a>
## 2 - 极深神经网络的问题

上周，你构建了第一个卷积神经网络：首先使用 NumPy 手动构建，然后使用 TensorFlow 和 Keras。 

近年来，神经网络变得更加深入，最先进的网络从只有几层（例如AlexNet）发展到一百多个层。

* 非常深的网络的主要好处是它可以表示非常复杂的函数。它还可以学习许多不同抽象级别的特征，从边缘（在较浅的层，更接近输入）到非常复杂的特征（在更深的层，更接近输出）。 

* 然而，使用更深的网络并不总是有帮助。训练它们的一个巨大障碍是梯度消失：非常深的网络通常具有很快变为零的梯度信号，从而使梯度下降速度过慢。

* 更具体地说，在梯度下降期间，当你从最后一层反向传播到第一层时，你将在每一步乘以权重矩阵，因此梯度可以呈指数快速下降到零（或者，在极少数情况下，快速呈指数增长并因获得非常大的值而“爆炸”）。 

* 在训练过程中，你可能会看到随着训练的进行，较浅层的梯度的幅度（或范数）很快降至零，如下所示： 

<img src="images/vanishing_grad_kiank.png" style="width:450px;height:220px;">
<div style="text-align:center;color:purple;"><u>  <b>图1</b> </u>： <b>梯度消失</b> <br> 当网络训练时，较浅层的学习速度下降得非常快</div>

别担心！你现在将通过构建残差网络来解决这个问题！

<a name='3'></a>
## 3 - 构建残差网络

在 ResNets 中，“快捷方式”或“跳跃连接”允许模型跳过层：  

<img src="images/skip_connection_kiank.png" style="width:650px;height:200px;">
<div style="text-align:center;color:purple;"><u>  <b>图2</b> </u>：显示跳跃连接的ResNet块<br></div>

左图显示了通过网络的“主要路径”。右侧的图像添加了主路径的快捷方式。通过将这些 ResNet 块堆叠在一起，你可以形成一个非常深的网络。 

讲座中提到，使用带有快捷方式的 ResNet 块也使得其中一个块很容易学习恒等映射。这意味着你可以堆叠额外的 ResNet 块，而损害训练集性能的风险很小。  
    
在这一点上，还有一些证据表明，学习恒等映射的容易性对 ResNets 卓越性能的贡献甚至超过了跳跃连接对梯度消失的帮助。

ResNet 中使用两种主要类型的块，主要取决于输入/输出维度是否相同或不同。你将实现这两个：“恒等块”和“卷积块”。

<a name='3-1'></a>
### 3.1 - 恒等块

恒等块是 ResNets 中使用的标准块，对应于输入激活值（例如 $a^{[l]}$）与输出激活值（例如 $a^{[l+2]}$）具有相同维度的情况。为了充实 ResNet 恒等块中发生的不同步骤，下面是显示各个步骤的替代图：

<img src="images/idblock2_kiank.png" style="width:650px;height:150px;">
<div style="text-align:center;color:purple;"><u>  <b>图3</b> </u>： <b>恒等块。</b> 跳跃连接“跳过”2层。</div>

上面的路径是“快捷路径”。较低的路径是“主路径”。在此图中，请注意每层中的 CONV2D 和 ReLU 步骤。为了加快训练速度，添加了 BatchNorm 步骤。不用担心实现起来很复杂——你会看到 BatchNorm 只是 Keras 中的一行代码！ 

在本练习中，你实际上将实现恒等块的稍微更强大的版本，其中跳跃连接 “跳过”3 个隐藏层而不是 2 个层。它看起来像这样： 

<img src="images/idblock3_kiank.png" style="width:650px;height:150px;">
<div style="text-align:center;color:purple;"><u>  <b>图4</b> </u>： <b>恒等块。</b> 跳跃连接“跳过”3层。</div>

以下是各个步骤：

主路径的第一个组成部分： 
- 第一个 CONV2D 具有形状 (1,1) 和步长 (1,1) 的 $F_1$ 卷积核。它的填充是'valid'。使用 0 作为随机均匀初始化的种子：`kernel_initializer = initializer(seed=0)`。 
- 第一个 BatchNorm 是归一化 “通道”轴。
- 然后应用ReLU 激活函数。这没有超参数。 

主路径的第二个组成部分：
- 第二个 CONV2D 具有形状为 $(f,f)$ 且步长为 (1,1) 的 $F_2$ 卷积核。它的填充是'same'。使用 0 作为随机均匀初始化的种子：`kernel_initializer = initializer(seed=0)`。
- 第二个 BatchNorm 是归一化 “通道”轴。
- 然后应用ReLU 激活函数。这没有超参数。

主路径的第三个组成部分：
- 第三个 CONV2D 具有形状 (1,1) 和步长 (1,1) 的 $F_3$ 卷积核。它的填充是'valid'。使用 0 作为随机均匀初始化的种子：`kernel_initializer = initializer(seed=0)`。 
- 第三个 BatchNorm 是归一化 “通道”轴。
- 请注意，该组件中**没有** ReLU 激活函数。 

最后一步： 
- `X_shortcut` 和第 3 层 `X` 的输出相加。
- **提示**：语法看起来像 `Add()([var1,var2])`
- 然后应用ReLU 激活函数。这没有超参数。 

<a name='ex-1'></a>
### 练习 1 - identity_block

实现 ResNet 恒等块。主路径的第一个组件已经为你实现了！首先，你应该仔细阅读这些文档，以确保你了解正在发生的事情。然后，实现其余部分。 
- 实现Conv2D步骤：[Conv2D](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Conv2D)
- 实现 BatchNorm：[BatchNormalization](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/BatchNormalization) `BatchNormalization(axis = 3)(X)`。如果训练设置为 False，则其权重不会使用新示例进行更新。当模型用于预测模式时为 I.e。
- 对于激活值，请使用：`Activation('relu')(X)`
- 使用 [Add](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Add) 将主路径的结果与快捷连接传递的值相加。

我们已将初始化参数添加到函数中。此参数接收初始化程序函数，如包 [tensorflow.keras.initializers](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/initializers) 中包含的初始化程序或任何其他自定义初始化程序。默认情况下，它将设置为 [random_uniform](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/initializers/RandomUniform)

请记住，这些函数接受 `seed` 参数，该参数可以是你想要的任何值，但在此 Notebook 必须设置为 0 以用于**评分目的**。

 这里是你实际使用Functional API 的强大功能来创建快捷路径的地方： 

In [ ]:
# UNQ_C1
# GRADED FUNCTION: identity_block

def identity_block(X, f, filters, initializer=random_uniform):
    """
    图 4 中定义的恒等块的实现
    
    参数：
    X -- 形状为 (m, n_H_prev, n_W_prev, n_C_prev) 的输入张量
    f -- 整数，指定主路径中间CONV窗口的形状
    filters -- python 整数列表，定义主路径 CONV 层中卷积核的数量
    initializer -- 设置层的初始权重。等于随机均匀初始值设定项
    
    返回：
    X -- 恒等块的输出，形状为 (m, n_H, n_W, n_C) 的张量
    """
    
    # 检索卷积核
    F1, F2, F3 = filters
    
    # 保存输入值。稍后你将需要它来添加回主路径。 
    X_shortcut = X
    
    # 主路径的第一个组成部分
    X = Conv2D(filters = F1, kernel_size = 1, strides = (1,1), padding = 'valid', kernel_initializer = initializer(seed=0))(X)
    X = BatchNormalization(axis = 3)(X) # 默认轴
    X = Activation('relu')(X)
    
    ### START CODE HERE
    ## 主路径的第二个组成部分（约 3 行）
    ## 设置填充 = 'same'
    X = None
    X = None
    X = None 

    ## 主路径的第三个组成部分（约2行）
    ## 设置填充 = 'valid'
    X = None
    X = None 
    
    ## 最后一步：将快捷方式值添加到主路径，并将其通过 ReLU 激活函数 （约 2 行）传递
    X = None
    X = None 
    ### END CODE HERE

    return X

In [ ]:
### 你无法编辑此单元格

tf.keras.backend.set_learning_phase(False)

np.random.seed(1)
tf.random.set_seed(2)
X1 = np.ones((1, 4, 4, 3)) * -1
X2 = np.ones((1, 4, 4, 3)) * 1
X3 = np.ones((1, 4, 4, 3)) * 3

X = np.concatenate((X1, X2, X3), axis = 0).astype(np.float32)

A3 = identity_block(X, f=2, filters=[4, 4, 3],
                   initializer=lambda seed=0:constant(value=1))
print('\033[1mWith training=False\033[0m\n')
A3np = A3.numpy()
print(np.around(A3.numpy()[:,(0,-1),:,:].mean(axis = 3), 5))
resume = A3np[:,(0,-1),:,:].mean(axis = 3)
print(resume[1, 1, 0])

tf.keras.backend.set_learning_phase(True)

print('\n\033[1mWith training=True\033[0m\n')
np.random.seed(1)
tf.random.set_seed(2)
A4 = identity_block(X, f=2, filters=[3, 3, 3],
                   initializer=lambda seed=0:constant(value=1))
print(np.around(A4.numpy()[:,(0,-1),:,:].mean(axis = 3), 5))

public_tests.identity_block_test(identity_block)

**预期值**

```
With training=False

[[[  0.        0.        0.        0.     ]
  [  0.        0.        0.        0.     ]]

 [[192.99992 192.99992 192.99992  96.99996]
  [ 96.99996  96.99996  96.99996  48.99998]]

 [[578.99976 578.99976 578.99976 290.99988]
  [290.99988 290.99988 290.99988 146.99994]]]
96.99996

With training=True

[[[0.      0.      0.      0.     ]
  [0.      0.      0.      0.     ]]

 [[0.40732 0.40732 0.40732 0.40732]
  [0.40732 0.40732 0.40732 0.40732]]

 [[5.00011 5.00011 5.00011 3.25955]
  [3.25955 3.25955 3.25955 2.40732]]]
[[[  0.        0.        0.        0.     ]
  [  0.        0.        0.        0.     ]]
All tests passed!
```

<a name='3-2'></a>
### 3.2 - 卷积块

ResNet“卷积块”是第二种块类型。当输入和输出尺寸不匹配时，你可以使用此类块。与恒等块的区别在于快捷路径中多了一个CONV2D层： 

<img src="images/convblock_kiank.png" style="width:650px;height:150px;">
<div style="text-align:center;color:purple;"><u>  <b>图4</b> </u>： <b>卷积块</b></div>

* 快捷路径中的 CONV2D 层用于将输入 $x$ 调整为不同的尺寸，以便尺寸在将快捷值添加回主路径所需的最终添加中匹配。 （这与讲座中讨论的矩阵 $W_s$ 的作用类似。） 
* 例如，要将激活值尺寸的高度和宽度减小 2 倍，可以使用步长为 2 的 1x1 卷积。 
* 快捷路径上的CONV2D层不使用任何非线性激活函数。它的主要作用是应用（学习的）线性函数来减少输入的维度，以便维度与后面的加法步骤相匹配。 
* 对于之前的练习，出于评分目的，需要额外的 `initializer` 参数，并且默认设置为 [glorot_uniform](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/initializers/GlorotUniform)

卷积块的详细信息如下。 

主路径的第一个组成部分：
- 第一个 CONV2D 具有形状 (1,1) 和步长 (s,s) 的 $F_1$ 卷积核。它的填充是'valid'。使用 0 作为 `glorot_uniform` 种子 `kernel_initializer = initializer(seed=0)`。
- 第一个 BatchNorm 是归一化 “通道”轴。
- 然后应用ReLU 激活函数。这没有超参数。 

主路径的第二个组成部分：
- 第二个 CONV2D 具有形状 (f,f) 和步长 (1,1) 的 $F_2$ 卷积核。它的填充是'same'。  使用 0 作为 `glorot_uniform` 种子 `kernel_initializer = initializer(seed=0)`。
- 第二个 BatchNorm 是归一化 “通道”轴。
- 然后应用ReLU 激活函数。这没有超参数。 

主路径的第三个组成部分：
- 第三个 CONV2D 具有形状为 (1,1) 且步长为 (1,1) 的 $F_3$ 卷积核。它的填充是'valid'。  使用 0 作为 `glorot_uniform` 种子 `kernel_initializer = initializer(seed=0)`。
- 第三个 BatchNorm 是归一化 “通道”轴。请注意，该组件中没有 ReLU 激活函数。 

快捷路径：
- CONV2D 具有形状为 (1,1) 且步长为 (s,s) 的 $F_3$ 卷积核。它的填充是'valid'。  使用 0 作为 `glorot_uniform` 种子 `kernel_initializer = initializer(seed=0)`。
- BatchNorm 是归一化 “通道”轴。 

最后一步： 
- 快捷方式和主路径值相加。
- 然后应用ReLU 激活函数。这没有超参数。 
 
<a name='ex-2'></a>    
### 练习 2 - convolutional_block
    
实现卷积块。主路径的第一个组件已经实现；然后就轮到你来实现剩下的了！和以前一样，始终使用 0 作为随机初始化的种子，以确保与评分器的一致性。
- [Conv2D](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Conv2D)
- [BatchNormalization](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/BatchNormalization)（轴：整数，应该是归一化后的（通常是特征轴）的轴）`BatchNormalization(axis = 3)(X)`。如果训练设置为 False，则其权重不会使用新示例进行更新。 I.e 当模型用于预测模式时。
- 对于激活值，请使用：`Activation('relu')(X)`
- [Add](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Add)
    
我们已将初始化参数添加到函数中。此参数接收初始化程序函数，如包 [tensorflow.keras.initializers](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/initializers) 中包含的初始化程序或任何其他自定义初始化程序。默认情况下，它将设置为 [glorot_uniform](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/initializers/GlorotUniform)

请记住，这些函数接受 `seed` 参数，该参数可以是你想要的任何值，但在此 Notebook 必须设置为 0 以用于**评分目的**。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: convolutional_block

def convolutional_block(X, f, filters, s = 2, initializer=glorot_uniform):
    """
    图 4 中定义的卷积块的实现
    
    参数：
    X -- 形状为 (m, n_H_prev, n_W_prev, n_C_prev) 的输入张量
    f -- 整数，指定主路径中间CONV窗口的形状
    filters -- python 整数列表，定义主路径 CONV 层中卷积核的数量
    s -- 整数，指定要使用的步长
    initializer -- 设置层的初始权重。等于 Glorot 统一初始值设定项， 
                   也称为 Xavier 统一初始值设定项。
    
    返回：
    X -- 卷积块的输出，形状为 (m, n_H, n_W, n_C) 的张量
    """
    
    # 检索卷积核
    F1, F2, F3 = filters
    
    # 保存输入值
    X_shortcut = X


    ##### 主路径#####
    
    # 主路径的第一个组成部分 glorot_uniform(seed=0)
    X = Conv2D(filters = F1, kernel_size = 1, strides = (s, s), padding='valid', kernel_initializer = initializer(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)

    ### START CODE HERE
    
    ## 主路径的第二个组成部分（约 3 行）
    X = None 
    X = None 
    X = None 

    ## 主路径的第三个组成部分（约2行）
    X = None 
    X = None 
    
    ##### 快捷路径 #####（约 2 行）
    X_shortcut = None
    X_shortcut = None
    
    ### END CODE HERE

    # 最后一步：将快捷方式值添加到主路径（使用此顺序 [X, X_shortcut]），并将其通过 ReLU 激活函数传递
    X = Add()([X, X_shortcut])
    X = Activation('relu')(X)
    
    return X
    

In [ ]:
### 你无法编辑此单元格

public_tests.convolutional_block_test(convolutional_block)

**预期值**

```
tf.Tensor(
[[[0.33485505 1.6415989  0.33789736 0.08511472 0.814965   0.        ]
  [0.17509979 1.5699672  0.2606045  0.         0.767209   0.        ]]

 [[0.         1.4983511  0.16896994 0.         0.61830646 0.        ]
  [0.         1.4502985  0.11632714 0.         0.58068544 0.        ]]], shape=(2, 2, 6), dtype=float32)
All tests passed!
```

<a name='4'></a>  
## 4 - 构建你的第一个 ResNet 模型（50 层）

你现在拥有构建非常深的 ResNet 所需的块。下图详细描述了该神经网络的架构。图中的“ID BLOCK”代表“恒等块”，“ID BLOCK x3”表示你应该将 3 个恒等块堆叠在一起。

<img src="images/resnet_kiank.png" style="width:850px;height:150px;">
<div style="text-align:center;color:purple;"><u>  <b>图5</b> </u>： <b>ResNet-50 模型</b></div>

这个ResNet-50模型的详细信息是：
- 零填充用 (3,3) 填充输入
- 第一阶段：
    - 2D 卷积有 64 个形状为 (7,7) 的卷积核，并使用 (2,2) 的步长。 
    - BatchNorm 应用于输入的“通道”轴。
    - MaxPooling 使用 (3,3) 窗口和 (2,2) 步长。
- 第二阶段：
    - 卷积块使用三组大小为 [64,64,256] 的卷积核，“f”为 3，“s”为 1。
    - 2个恒等块使用三组大小为[64,64,256]的卷积核，“f”为3。
- 第三阶段：
    - 卷积块使用三组大小为 [128,128,512] 的卷积核，“f”为 3，“s”为 2。
    - 3 个恒等块使用三组大小为 [128,128,512] 的卷积核，“f”为 3。
- 第四阶段：
    - 卷积块使用三组大小为 [256, 256, 1024] 的卷积核，“f”为 3，“s”为 2。
    - 5 个恒等块使用三组大小为 [256, 256, 1024] 的卷积核，“f”为 3。
- 第五阶段：
    - 卷积块使用三组大小为 [512, 512, 2048] 的卷积核，“f”为 3，“s”为 2。
    - 2 个恒等块使用三组大小为 [512, 512, 2048] 的卷积核，“f”为 3。
- 2D 平均池化使用形状为 (2,2) 的窗口 (pool_size)。
- “展平”层没有任何超参数。
- 全连接（全连接）层使用 softmax 激活函数将其输入减少到类的数量。

    
<a name='ex-3'></a>      
### 练习 3 - ResNet50 
    
实现上图所示的 50 层 ResNet。我们已经实现了第一阶段和第二阶段。请实现其余部分。 （实现阶段 3-5 的语法应该与阶段 2 的语法非常相似）确保遵循上面文本中的命名约定。 

你需要使用这个函数： 
- 平均池化 [参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/AveragePooling2D)

这是我们在下面的代码中使用的其他一些函数：
- Conv2D：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Conv2D)
- BatchNorm：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/BatchNormalization)（轴：Integer，应该是归一化后的（通常是特征轴）的轴）
- 零填充：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/ZeroPadding2D)
- 最大池化：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/MaxPool2D)
- 全连接层：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Dense)
- 添加：[参见参考资料](https://www.tensorflow.org/versions/r2.9/api_docs/python/tf/keras/layers/Add)

In [ ]:
# UNQ_C3
# GRADED FUNCTION: ResNet50

def ResNet50(input_shape = (64, 64, 3), classes = 6, training=False):
    """
    流行的 ResNet50 架构的分阶段实现：
    CONV2D -> BATCHNORM -> ReLU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL -> 展平 -> DENSE 

    参数：
    input_shape -- 数据集图像的形状
    classes -- 整数，类数

    返回：
    model -- Keras中的Model()实例
    """
    
    # 将输入定义为形状为 input_shape 的张量
    X_input = Input(input_shape)

    
    # 零填充
    X = ZeroPadding2D((3, 3))(X_input)
    
    # 第一阶段
    X = Conv2D(64, (7, 7), strides = (2, 2), kernel_initializer = glorot_uniform(seed=0))(X)
    X = BatchNormalization(axis = 3)(X)
    X = Activation('relu')(X)
    X = MaxPooling2D((3, 3), strides=(2, 2))(X)

    # 第二阶段
    X = convolutional_block(X, f = 3, filters = [64, 64, 256], s = 1)
    X = identity_block(X, 3, [64, 64, 256])
    X = identity_block(X, 3, [64, 64, 256])

    ### START CODE HERE
    
    # 使用上面的说明来实现下面的所有阶段
    # 确保你没有错过添加任何必需的参数
    
    ## 第 3 阶段（约 4 行）
    # `convolutional_block`，此阶段的正确值为 `f`、`filters` 和 `s`
    X = None
    
    # 3 个 `identity_block`，此阶段的 `f` 和 `filters` 的正确值
    X = None
    X = None
    X = None

    # 第 4 阶段（约 6 行）
    # 为此阶段添加 `convolutional_block` 以及正确的 `f`、`filters` 和 `s` 值
    X = None
    
    # 5 个 `identity_block`，此阶段的 `f` 和 `filters` 的正确值
    X = None
    X = None
    X = None
    X = None
    X = None

    # 第5阶段（约3行）
    # 为此阶段添加 `convolutional_block` 以及正确的 `f`、`filters` 和 `s` 值
    X = None
    
    # 2 个 `identity_block`，此阶段的正确值为 `f` 和 `filters`
    X = None
    X = None

    # AVGPOOL（约 1 行）。使用"X = AveragePooling2D()(X)"
    X = None
    
    ### END CODE HERE

    # 输出层
    X = Flatten()(X)
    X = Dense(classes, activation='softmax', kernel_initializer = glorot_uniform(seed=0))(X)
    
    
    # 创建模型
    model = Model(inputs = X_input, outputs = X)

    return model

运行以下代码来构建模型的图。如果你的实现不正确，你可以通过在运行下面的 `model.fit(...)` 时检查你的准确性来了解这一点。

In [ ]:
tf.keras.backend.set_learning_phase(True)

model = ResNet50(input_shape = (64, 64, 3), classes = 6)
print(model.summary())

In [ ]:
### 你无法编辑此单元格

from outputs import ResNet50_summary

model = ResNet50(input_shape = (64, 64, 3), classes = 6)

comparator(summary(model), ResNet50_summary)


如Keras教程Notebook所示，在训练模型之前，你需要通过编译模型来配置学习过程。

In [ ]:
np.random.seed(1)
tf.random.set_seed(2)
opt = tf.keras.optimizers.Adam(learning_rate=0.00015)
model.compile(optimizer=opt, loss='categorical_crossentropy', metrics=['accuracy'])

模型现在已准备好接受训练。你现在唯一需要的是数据集！

让我们加载你的老朋友 SIGNS 数据集。

<img src="images/signs_data_kiank.png" style="width:450px;height:250px;">
<div style="text-align:center;color:purple;"><u>  <b>图 6</b> </u> : <b>SIGNS 数据集</b></div>


In [ ]:
X_train_orig, Y_train_orig, X_test_orig, Y_test_orig, classes = load_dataset()

# 归一化图像载体
X_train = X_train_orig / 255.
X_test = X_test_orig / 255.

# 将训练和测试标签转换为 one-hot 矩阵
Y_train = convert_to_one_hot(Y_train_orig, 6).T
Y_test = convert_to_one_hot(Y_test_orig, 6).T

print ("number of training examples = " + str(X_train.shape[0]))
print ("number of test examples = " + str(X_test.shape[0]))
print ("X_train shape: " + str(X_train.shape))
print ("Y_train shape: " + str(Y_train.shape))
print ("X_test shape: " + str(X_test.shape))
print ("Y_test shape: " + str(Y_test.shape))

运行以下单元格，以批量大小为 32 的 10 个周期训练模型。在 GPU 上，该过程应该需要不到 2 分钟。 

In [ ]:
model.fit(X_train, Y_train, epochs = 10, batch_size = 32)

**预期输出**：

```
Epoch 1/10
34/34 [==============================] - 16s 64ms/step - loss: 1.7770 - accuracy: 0.3111
Epoch 2/10
34/34 [==============================] - 2s 50ms/step - loss: 1.1800 - accuracy: 0.5583
Epoch 3/10
34/34 [==============================] - 2s 51ms/step - loss: 0.7900 - accuracy: 0.6935
Epoch 4/10
34/34 [==============================] - 2s 50ms/step - loss: 0.5295 - accuracy: 0.8065
Epoch 5/10
34/34 [==============================] - 2s 50ms/step - loss: 0.3665 - accuracy: 0.8648
Epoch 6/10
34/34 [==============================] - 2s 50ms/step - loss: 0.3032 - accuracy: 0.8880
Epoch 7/10
34/34 [==============================] - 2s 51ms/step - loss: 0.2456 - accuracy: 0.9194
Epoch 8/10
34/34 [==============================] - 2s 51ms/step - loss: 0.2123 - accuracy: 0.9278
Epoch 9/10
34/34 [==============================] - 2s 50ms/step - loss: 0.2113 - accuracy: 0.9389
Epoch 10/10
34/34 [==============================] - 2s 50ms/step - loss: 0.1469 - accuracy: 0.9491
```

确切的值无法匹配，但不用担心。你必须看到的重要一点是，前 5 个 epoch 的损失值降低了，而准确率提高了。

让我们看看这个模型（仅训练了两个 epoch）在测试集上的表现如何。

In [ ]:
preds = model.evaluate(X_test, Y_test)
print ("Loss = " + str(preds[0]))
print ("Test Accuracy = " + str(preds[1]))

**预期输出**：

<table>
    <tr>
        <td>
            <b>测试精度</b>
        </td>
        <td>
           >0.70
        </td>
    </tr>

</table>

出于此作业的目的，你被要求对模型进行十个 epoch 的训练。你可以看到它的性能很好。在线评分器也只会在少数epoch运行你的代码。请继续提交你的作业。 

完成作业的官方（计分）部分后，如果需要，你还可以选择训练 ResNet 以进行更多迭代。当训练大约 20 个 epoch 时，它往往会获得更好的性能，但在 CPU 上训练时确实需要一个多小时。 

使用 GPU，此 ResNet50 模型的权重在 SIGNS 数据集上进行训练。你可以在下面的单元格中的测试集上加载并运行经过训练的模型。加载模型可能需要约 1 分钟。玩得开心！ 

In [ ]:
pre_trained_model = load_model('resnet50.h5')

In [ ]:
preds = pre_trained_model.evaluate(X_test, Y_test)
print ("Loss = " + str(preds[0]))
print ("Test Accuracy = " + str(preds[1]))

**恭喜**完成此作业！你现在已经实现了最先进的图像分类系统！呜呼！ 

ResNet50 经过足够次数的迭代训练后，是一种强大的图像分类模型。希望从现在开始，你可以使用所学到的知识并将其应用到你自己的分类问题中，以实现最先进的准确性。

<div style="color:blue;">

**你应该记住什么**：

- 非常深的“普通”网络在实践中不起作用，因为梯度消失使它们难以训练。  
- 跳跃连接帮助解决消失的梯度问题。它们还使 ResNet 块可以轻松学习恒等映射。 
- 有两种主要类型的块： **恒等块** 和 **卷积块**。 
- 非常深的残差网络是通过将这些块堆叠在一起构建的。

</div>

## 为其他学习者释放资源

如果你不打算继续学习下一个 `Optional` 部分，请通过运行下面的单元格来释放你的作业使用的资源，以便其他学习者可以像你一样利用这些资源，从而帮助我们为学习者提供流畅的学习体验。谢谢你！

**注意**： 
- 完成作业并准备好提交进行评分后，运行下面的单元格。
- 运行它时，将打开一个弹出窗口，单击“`Ok`”。
- 运行单元将是 `restart the kernel`。

In [ ]:
# %%javascript
# IPython.notebook.save_checkpoint();
# if (confirm("Clear memory?") == true)
# {
#     IPython.notebook.kernel.restart();
# }

<a name='5'></a>  
## 5 - 在你自己的图像上进行测试（可选/不计分）

如果你愿意，你还可以拍一张自己的手的照片并查看模型的输出。为此：
    1. 单击此 Notebook 上部栏中的“文件”，然后单击“打开”以转到你的 Coursera Hub。
    2. 将图像添加到 Jupyter Notebook 目录的“images”文件夹中
    3. 在下面的代码中写下你的图片名称
    4. 运行代码并检查算法是否正确！ 

In [ ]:
img_path = 'images/my_image.jpg'
img = image.load_img(img_path, target_size=(64, 64))
x = image.img_to_array(img)
x = np.expand_dims(x, axis=0)
x = x/255.0
x2 = x 
print('Input image shape:', x.shape)
imshow(img)
prediction = pre_trained_model.predict(x2)
print("Class prediction vector [p(0), p(1), p(2), p(3), p(4), p(5)] = ", prediction)
print("Class:", np.argmax(prediction))


尽管模型具有很高的准确率，但它在你自己的图像集上可能表现不佳。请注意，图片的形状、拍摄照片的照明以及所有预处理步骤都会对模型的性能产生影响。考虑到到目前为止你在本专项课程中学到的所有内容，你认为原因可能是什么？

*提示*：这可能与某些发行版有关。你能想出一个潜在的解决方案吗？

你还可以通过运行以下代码来打印模型的摘要。

In [ ]:
pre_trained_model.summary()

## 为其他学习者释放资源

为了给我们的学习者提供流畅的学习体验，请通过运行下面的单元格来释放你的作业使用的资源，以便其他学习者可以像你一样利用这些资源。谢谢你！

**注意**： 
- 完成作业并准备好提交进行评分后，运行下面的单元格。
- 运行它时，将打开一个弹出窗口，单击“`Ok`”。
- 运行单元将是 `restart the kernel`。

In [ ]:
%%javascript
IPython.notebook.save_checkpoint();
if (confirm("Clear memory?") == true)
{
    IPython.notebook.kernel.restart();
}

<a name='6'></a>  
## 6 - 参考文献

这个 Notebook 展示了 He 等人的 ResNet 算法。 （2015）。这里的实现也受到了重要的启发，并遵循 Francois Chollet 的 GitHub 存储库中给出的结构： 

- 何凯明、张翔宇、任少卿、孙健 - [用于图像识别的深度残差学习 (2015)](https://arxiv.org/abs/1512.03385)
- Francois Chollet 的 GitHub 存储库：https://github.com/fchollet/deep-learning-models/blob/master/resnet50.py
